In [ ]:
import pandas as pd
import glob
import os
from tqdm.auto import tqdm
import geopandas as gpd
from shapely import wkt
from math import ceil
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
from datetime import timedelta, time
import datetime
import re

In [ ]:
tqdm.pandas()

In [ ]:
def generate_timeframes_vectorized(row):
    # Generate full range for the entire interval
    all_times = pd.date_range(start=row['FINI'], end=row['FFIN'], freq='15T')

    # Convert to array of times for filtering
    times_of_day = all_times.time

    # Handle normal vs overnight HCIE
    if row['hcie_start'] <= row['hcie_end']:
        mask = (times_of_day >= row['hcie_start']) & (times_of_day <= row['hcie_end'])
    else:
        mask = (times_of_day >= row['hcie_start']) | (times_of_day <= row['hcie_end'])

    # Apply mask and return filtered slots
    return all_times[mask]

In [ ]:
mvi = gpd.read_file("../datos/2_Estructura_via/estructura_via_completa/MVI.geojson")
mvi = mvi.to_crs("EPSG:4326")
mvi.columns = mvi.columns.str.lower()
mvi["id_mvi"] = mvi["mviciv"].astype(str) + "_" + mvi["mvisvia"].str.replace(" ", "")
mvi['contador'] = mvi.groupby('id_mvi').cumcount() + 1
mvi['id_mvi'] = mvi['id_mvi'] + '.' + mvi['contador'].astype(str)

buffer_distance = 4
mvi_buffered = mvi.to_crs("EPSG:3116").copy()
mvi_buffered['mvi_length'] = mvi_buffered.geometry.length
mvi_buffered['geometry'] = mvi_buffered.geometry.buffer(buffer_distance, cap_style="square")
mvi_buffered_buffer = mvi_buffered.set_geometry('geometry')
mvi_buffered_buffer = mvi_buffered_buffer.to_crs("EPSG:4326")

In [ ]:
file_output_name = "../datos_procesados/pmts_processed/"
# Crear carpeta de salida si no existe
os.makedirs(file_output_name, exist_ok=True)

In [ ]:
coi_parques = gpd.read_file('../datos/9_pmts/COI_PARQUES_202504071925.geojson')
coi_parques = coi_parques.to_crs("EPSG:4326")
coi_parques_intersections = gpd.overlay(coi_parques, mvi_buffered_buffer[['id_mvi', 'mvi_length', 'geometry']], how='intersection')
coi_parques_intersections = coi_parques_intersections.to_crs("EPSG:3116").copy()
coi_parques_intersections['intersection_length'] = coi_parques_intersections.geometry.length
coi_parques_intersections['overlap_ratio'] = coi_parques_intersections['intersection_length'] / coi_parques_intersections['mvi_length']
coi_parques_intersections = coi_parques_intersections[coi_parques_intersections['overlap_ratio'] > 0.15]
coi_parques_intersections = coi_parques_intersections.to_crs("EPSG:4326").copy()
coi_parques_intersections['road_id'] = coi_parques_intersections['id_mvi']
coi_parques_intersections['coi_parques'] = 1
coi_parques_work = coi_parques_intersections[['road_id','FINI', 'FFIN','HCIE','coi_parques']].copy()
hcie_mapping = {
    "24 HORAS": (time(0, 0), time(23, 59)),
    "06:00-18:00": (time(6, 0), time(18, 0)),
    "07:00-18:00": (time(7, 0), time(18, 0))
}
coi_parques_work["hcie_start"], coi_parques_work["hcie_end"] = zip(*coi_parques_work["HCIE"].map(hcie_mapping))
coi_parques_work['time_slots'] = coi_parques_work.progress_apply(generate_timeframes_vectorized, axis=1)
coi_parques_work = coi_parques_work.explode('time_slots').reset_index(drop=True)
coi_parques_work['timeframe'] = coi_parques_work['time_slots'].dt.strftime('%Y-%m-%d %H:%M:%S')
coi_parques_dataset = coi_parques_work[['road_id','timeframe','coi_parques']]

In [ ]:
coi_parques_work.HCIE.unique()

In [ ]:

coi_parques_dataset.to_csv(f'{file_output_name}coi_parques_dataset.csv', index=False)

In [ ]:
coi_puntos.HCIE.unique()

In [ ]:
coi_puntos = gpd.read_file('../data/9_pmts/COI_PUNTO_202504071914.geojson')

In [ ]:
WEEKDAY_MAP = {
    "LUNES": 0, "MARTES": 1, "MIERCOLES": 2, "MIÉRCOLES": 2,
    "JUEVES": 3, "VIERNES": 4, "SABADO": 5, "SÁBADO": 5, "DOMINGO": 6
}

# 2️⃣ Normalization function
def normalize_hcie_text(text):
    text = str(text).upper()
    text = re.sub(r'\s+', ' ', text)  # Normalize spaces
    text = text.replace('L A V', 'LUNES A VIERNES')
    text = text.replace('L-V', 'LUNES A VIERNES')
    text = text.replace('L-VIERNES', 'LUNES A VIERNES')
    text = text.replace('L-VI', 'LUNES A VIERNES')
    text = text.replace('SAB', 'SABADO')
    text = text.replace('DOM', 'DOMINGO')
    text = text.replace('FESTIVOS', 'DOMINGO')
    text = text.replace('Á', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U')
    text = text.replace(' Y ', ' ')
    text = text.strip()
    return text

# 3️⃣ Extract list of weekdays from text block
def extract_weekdays(block):
    weekdays = set()

    # Generic day range extraction (fully dynamic)
    range_match = re.search(
        r'(LUNES|MARTES|MIERCOLES|JUEVES|VIERNES|SABADO|DOMINGO)\s+A\s+(LUNES|MARTES|MIERCOLES|JUEVES|VIERNES|SABADO|DOMINGO)',
        block
    )
    if range_match:
        start_day = WEEKDAY_MAP[range_match.group(1)]
        end_day = WEEKDAY_MAP[range_match.group(2)]
        if start_day <= end_day:
            weekdays.update(range(start_day, end_day + 1))
        else:
            weekdays.update(list(range(start_day, 7)) + list(range(0, end_day + 1)))

    # Also support single days not inside a range
    for word in WEEKDAY_MAP.keys():
        if word in block:
            weekdays.add(WEEKDAY_MAP[word])

    # If range_match already filled weekdays, avoid adding redundant single day matches:
    if range_match:
        return weekdays

    # Default fallback
    if not weekdays:
        weekdays.update(range(7))

    return weekdays

# 4️⃣ Extract timeframes from text block
def extract_timeframes(block):
    timeframes = []
    matches = re.findall(r"(\d{1,2}:\d{2})\s*[-–]\s*(\d{1,2}:\d{2})", block)
    for start_str, end_str in matches:
        try:
            start = datetime.datetime.strptime(start_str, "%H:%M").time()
            end = datetime.datetime.strptime(end_str, "%H:%M").time()
            timeframes.append((start, end))
        except:
            continue
    return timeframes

# 5️⃣ Full robust parser per row
def parse_hcie_row(raw_text):
    text = normalize_hcie_text(raw_text)

    # Special case: 24 horas applies to all days
    if '24 HORAS' in text:
        return {day: [(datetime.time(0, 0), datetime.time(23, 59))] for day in range(7)}

    # Split text into blocks (attempt splitting based on weekdays appearing)
    blocks = re.split(r'(LUNES|MARTES|MIERCOLES|JUEVES|VIERNES|SABADO|DOMINGO)', text)

    # Reconstruct the blocks keeping the keywords
    reconstructed = []
    i = 0
    while i < len(blocks):
        if blocks[i] in WEEKDAY_MAP:
            reconstructed.append(blocks[i] + ' ' + blocks[i+1])
            i += 2
        else:
            reconstructed.append(blocks[i])
            i += 1

    schedule = {day: [] for day in range(7)}

    for block in reconstructed:
        days = extract_weekdays(block)
        timeframes = extract_timeframes(block)
        for day in days:
            schedule[day].extend(timeframes)

    return schedule

In [ ]:
coi_puntos['parsed_schedule'] = coi_puntos['HCIE'].map(parse_hcie_row)

In [ ]:
coi_puntos['parsed_schedule']

In [ ]:
weekdays = set()

    # Generic day range extraction (fully dynamic)
range_match = re.search(
    r'(LUNES|MARTES|MIERCOLES|JUEVES|VIERNES|SABADO|DOMINGO)\s+A\s+(LUNES|MARTES|MIERCOLES|JUEVES|VIERNES|SABADO|DOMINGO)',
    block
)
if range_match:
    start_day = WEEKDAY_MAP[range_match.group(1)]
    end_day = WEEKDAY_MAP[range_match.group(2)]
    if start_day <= end_day:
        weekdays.update(range(start_day, end_day + 1))
    else:
        weekdays.update(list(range(start_day, 7)) + list(range(0, end_day + 1)))

# Also support single days not inside a range
for word in WEEKDAY_MAP.keys():
    if word in block:
        weekdays.add(WEEKDAY_MAP[word])

# If range_match already filled weekdays, avoid adding redundant single day matches:
if range_match:
    return weekdays

# Default fallback
if not weekdays:
    weekdays.update(range(7))

return weekdays

In [ ]:
range_match = re.search(
    r'(LUNES|MARTES|MIERCOLES|JUEVES|VIERNES|SABADO|DOMINGO)\s+A\s+(LUNES|MARTES|MIERCOLES|JUEVES|VIERNES|SABADO|DOMINGO)',
    'LUNES A SABADO 07:00-17:00'
)

In [ ]:
range_match.group(0)

In [ ]:
WEEKDAY_MAP[range_match.group(1)]

In [ ]:
range_match

In [ ]:
coi_puntos['HCIE_clean'] = coi_puntos['HCIE'].map(normalize_text)
coi_puntos['parsed_schedule'] = coi_puntos['HCIE'].map(parse_hcie_row)

In [ ]:
coi_puntos[['HCIE','parsed_schedule']]

In [ ]:
i = 10
coi_puntos.loc[i].HCIE, coi_puntos.loc[i].parsed_schedule

In [ ]:
list(coi_puntos.HCIE.unique())

In [ ]:
coi_puntos

In [ ]:
coi_tramo = gpd.read_file('../data/9_pmts/COI_TRAMO_202504071919.geojson')

In [ ]:
coi_tramo.columns

In [ ]:
coos_punto = gpd.read_file('../data/9_pmts/COOS_PUNTO_202504071922.geojson')

In [ ]:
coos_punto

In [ ]:
coos_punto = gpd.read_file('../data/9_pmts/COOS_PUNTO_202504071922.geojson')

In [ ]:
paraderos = gpd.read_file('../data/15_Paraderos_transp_publico_Zonales_SITP/Paraderos_Zonales_del_SITP.shp')

In [ ]:
paraderos.to_crs(epsg=4385)

In [ ]:
paraderos = paraderos.to_crs(epsg = 4326), 3116

In [ ]:
paraderos

In [ ]:
paraderos.head(40)